# Text Splitters and Chunking Process

The Big Picture
In Lecture 5, we learned how to load documents. But here's the problem:

A 100-page PDF becomes one giant wall of text.
An LLM can't process that. A search engine can't match against that.
We need to cut it into smart, bite-sized pieces — that's chunking.

Chunking is one of the most important decisions in any RAG system.
Bad chunks = bad retrieval = bad answers, no matter how good your LLM is.

What You Will Learn
#	Topic	Real-World Analogy
1	Why we chunk documents	---> Why you slice a pizza before serving 

2	Chunk size — the Goldilocks problem --->	Slices too small or too big

3	Overlap — why chunks share text	--->Pages in a book that repeat the last sentence

4	RecursiveCharacterTextSplitter --->	The smart pizza cutter

5	Other splitters --->	Specialized tools for special jobs

6	Metadata preservation   --->	Never lose track of where a chunk came from

7	Hands-on: split, compare, choose	--->Build it yourself!


---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.  
If you already have them from Lecture 5, you're good to go!

```python

In [3]:
%pip install langchain langchain-community langchain-text-splitters pypdf

Note: you may need to restart the kernel to use updated packages.


# 1. Why Do We Chunk Documents?
The Pizza Analogy
Imagine you ordered a pizza. The chef hands you the entire uncut pizza.
You can't eat it like that! You need to cut it into slices.

But HOW you cut it matters:
**
Too small (tiny squares)** — each piece has barely any topping, hard to enjoy
**Too big (half the pizza)** — too much to handle at once
**Just right (proper slices)** — each slice is a satisfying, complete portion
Documents work the same way. Here are the 3 reasons we chunk:

**Reason 1: Context Window Limits**
LLMs can only read a limited amount of text at once (the "context window").
You can't send 500 pages to an LLM — it will either crash or ignore most of it.

**Reason 2: Retrieval Precision**
When a user asks a question, you want to find the exact 2-3 paragraphs that
answer it — not dump 50 pages and hope the LLM figures it out.

**Reason 3: Semantic Coherence**
Each chunk should contain one complete idea. If you cut in the middle of a
sentence, the chunk becomes meaningless.

**Bottom line:** Bad chunking = bad RAG. No matter how good your LLM is,
if you feed it garbage chunks, you get garbage answers.

In [4]:
# lets load our sample article  
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path="data\\nlp.txt", encoding = 'utf-8')
documents = loader.load()

full_text = documents[0].page_content
print(f"Full text length: {len(full_text):,} characters")

print(f"Tha's roughly {len(full_text) // 4 :,} tokens")

print(f"\nFirst 300 characters:")
print(full_text[:300])



Full text length: 1,345 characters
Tha's roughly 336 tokens

First 300 characters:
Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.



In [5]:
sample_text = full_text[:1000]

chunk_size = [200, 500, 700]

for size in chunk_size:
    num_chunks = len(sample_text) // size 

    first_chunk = sample_text[:size]
   

print(f"\nchunk size: {size} characters")
print(f"Number of chunks from 1000 characters: {num_chunks} chunks")
print(f"First chunk ({len(first_chunk)} characters):")
print(first_chunk)




chunk size: 700 characters
Number of chunks from 1000 characters: 1 chunks
First chunk (700 characters):
Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.
NLP combines computational linguistics with machine learning and deep learning techniques. It is widely used in applications such as chatbots, machine translation, sentiment analysis, and information retrieval systems.

History of NLP
The history of NLP began in the 1950s with early rule-based systems. One of the first examples was the Georgetown-IBM experiment in , which demonstrated automatic tr


# OVERLAP 

In [6]:
# Let's visualize HOW overlap works with a simple example

demo_text = (
    "BERT was released by Google in 2018. "
    "It achieved state-of-the-art results on 11 NLP tasks. "
    "GPT was developed by OpenAI. "
    "It uses a different training approach called autoregressive modeling."
)

chunk_size = 70
overlap_size = 20

print(f"\nFull Text:({len(demo_text)} characters):'{demo_text}'")
print(f"\nchunk size: {chunk_size} characters | overlap size: {overlap_size} characters")
print("*" * 80)

step = chunk_size - overlap_size
chunk_number = 1
start = 0

while start < len(demo_text):
    end = start + chunk_size 
    chunk = demo_text[start:end]
    print(f"\nChunk {chunk_number} (char {start}-{min (end, len(demo_text))}): '{chunk}'")
    start += step
    chunk_number += 1



Full Text:(189 characters):'BERT was released by Google in 2018. It achieved state-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. It uses a different training approach called autoregressive modeling.'

chunk size: 70 characters | overlap size: 20 characters
********************************************************************************

Chunk 1 (char 0-70): 'BERT was released by Google in 2018. It achieved state-of-the-art resu'

Chunk 2 (char 50-120): 'tate-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. '

Chunk 3 (char 100-170): 'eveloped by OpenAI. It uses a different training approach called autor'

Chunk 4 (char 150-189): 'pproach called autoregressive modeling.'


# 4. RecursiveCharacterTextSplitter — The Smart Pizza Cutter
This is the #1 most commonly used splitter in LangChain, and your best default.

Why "Recursive"?
It tries to split text at natural boundaries, working through a priority list:

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 


splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=20) 

# pass the Document list (not the list itself) using split_documents
chunks = splitter.split_documents(documents)

print(f"Full text length: {len(full_text):,} characters")
print(f"After splitting: {len(chunks)} chunks")
print(f"settings: chunk_size=200, chunk_overlap=20")



Full text length: 1,345 characters
After splitting: 6 chunks
settings: chunk_size=200, chunk_overlap=20


In [8]:
# let see first 3 chunks

for i, chunk in enumerate(chunks[:3]):
    print("+" * 80)
    print(f"Chunk {i + 1} of {len(chunks)}")
    print("=" * 80)  
    print(f"Length: {len(chunk.page_content)} characters")
    print(f"Metadata: {chunk.metadata}")
    print(f"\nPage content")
    print(chunk.page_content)

  
    

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Chunk 1 of 6
Length: 299 characters
Metadata: {'source': 'data\\nlp.txt'}

Page content
Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Chunk 2 of 6
Length: 218 characters
Metadata: {'source': 'data\\nlp.txt'}

Page content
NLP combines computational linguistics with machine learning and deep learning techniques. It is widely used in applications such as chatbots, machine translation, sentiment analysis, and information retrieval systems.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Chunk 3 of 6
Length: 224 characters
Metadata: {'source': 'data

## Character Text Splitter

In [9]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(separator= "\n\n", chunk_size=300, chunk_overlap=20)

simple_chunks = splitter.split_documents(documents)

print(f"Full text length: {len(full_text):,} characters")

print(f"\nFirst chunks: {len(simple_chunks[0].page_content)} characters")
print(simple_chunks[0].page_content)


Created a chunk of size 518, which is longer than the specified 300
Created a chunk of size 441, which is longer than the specified 300


Full text length: 1,345 characters

First chunks: 518 characters
Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.
NLP combines computational linguistics with machine learning and deep learning techniques. It is widely used in applications such as chatbots, machine translation, sentiment analysis, and information retrieval systems.


## Token Text Splitter

In [12]:
%pip install tiktoken
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=300, chunk_overlap=20)
token_chunks = token_splitter.split_documents(documents)

print(f"Full text length: {len(token_chunks)} chunks")
print(f"\nFirst chunk: {len(token_chunks[0].page_content)}")
print(f"Content preview: {token_chunks[0].page_content}")


Note: you may need to restart the kernel to use updated packages.
Full text length: 1 chunks

First chunk: 1345
Content preview: Introduction to Natural Language Processing
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the interaction between computers and human language. The goal of NLP is to enable machines to understand, interpret, and generate human language in a valuable way.
NLP combines computational linguistics with machine learning and deep learning techniques. It is widely used in applications such as chatbots, machine translation, sentiment analysis, and information retrieval systems.

History of NLP
The history of NLP began in the 1950s with early rule-based systems. One of the first examples was the Georgetown-IBM experiment in , which demonstrated automatic translation of Russian sentences into English.
In the, statistical methods became popular. Researchers started using large corpora of text to train models. In recent years, 

## Markdown Text Splitter


In [32]:
from langchain_text_splitters import MarkdownTextSplitter

# laod a markdown file
md_loader = TextLoader(
    file_path="data\markdown_test_data.md", encoding = 'utf-8'
)
md_documents = md_loader.load()

print(f"Full markdown text length: {len(md_documents[0].page_content):,} characters")
print()

md_splitter = MarkdownTextSplitter(chunk_size=500, chunk_overlap=30)
md_chunks = md_splitter.split_documents(md_documents)

print(f"Markdown chunks created: {len(md_chunks)}")
print(f"\nFirst chunk: {len(md_chunks[0].page_content)} characters")
print(f"Content preview:\n{md_chunks[0].page_content}")
# to show full content from chunk 1 to 10
print(f"Full content of chunks 1-10:\n{chr(10).join([md_chunks[i].page_content for i in range(1, 11)])}")


Full markdown text length: 8,322 characters

Markdown chunks created: 24

First chunk: 51 characters
Content preview:
# Artificial Intelligence: A Comprehensive Overview
Full content of chunks 1-10:
Artificial Intelligence (AI) has rapidly transitioned from a theoretical computer science concept to a foundational technology driving modern innovation. It encompasses the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. 

## The Evolution of AI
## The Evolution of AI

The history of AI dates back to the mid-20th century. The term itself was coined in 1956 at the Dartmouth Conference, which is widely considered the birthplace of the AI field. Early AI research focused heavily on symbolic AI, where logic and rules were hard-coded into systems. This era saw the deve

# Metadata Preservation
When you split documents, you want to keep track of where each chunk came from.

In [40]:
from langchain_community.document_loaders import PyPDFLoader
# load a text file with metadata
pdf_loader = PyPDFLoader(file_path="data\Introduction to Natural Language Processing.pdf")
documents = pdf_loader.load()

print(f"origin documents :{len(documents)} loaded from pdf")
print(f"Page 1 content preview:\n{documents[0].page_content[:500]}")
print(f"page 1 metadata:\n{documents[0].metadata}")

#now split the documents into chunks while preserving metadata
pdf_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=20)
pdf_chunks = pdf_splitter.split_documents(documents)

print(f"Total chunks created:\n {len(pdf_chunks)}")
print(f"\nFirst chunk content preview:\n{pdf_chunks[0].page_content[:500]}")
print(f"\nFirst chunk metadata:\n{pdf_chunks[0].metadata}")



origin documents :3 loaded from pdf
Page 1 content preview:
Introduction to Natural Language Processing 
Natural Language Processing (NLP) is a subfield of Artificial Intelligence that focuses on the 
interaction between computers and human language. The goal of NLP is to enable machines to 
understand, interpret, and generate human language in a valuable way. 
NLP combines computational linguistics with machine learning and deep learning techniques. It 
is widely used in applications such as chatbots, machine translation, sentiment analysis, and 
inform
page 1 metadata:
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-04-07T19:19:51+05:00', 'author': 'Moorche', 'moddate': '2026-04-07T19:19:51+05:00', 'source': 'data\\Introduction to Natural Language Processing.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
Total chunks created:
 11

First chunk content preview:
Introduction to Natural Language Processing 
Natural Language Processing 